      -----------------------------------------------------------      
     |                   =====================                   |     
     |                           x T B                           |     
     |                   =====================                   |     
     |                         S. Grimme                         |     
     |          Mulliken Center for Theoretical Chemistry        |     
     |                    University of Bonn                     |     
      -----------------------------------------------------------      

   * xtb version 6.7.1 (edcfbbe) compiled by 'conda@c0858423619e' on 2025-09-04

   xtb is free software: you can redistribute it and/or modify it under
   the terms of the GNU Lesser General Public License as published by
   the Free Software Foundation, either version 3 of the License, or
   (at your option) any later version.
   
   xtb is distributed in the hope that it will be useful,
   but WITHOUT ANY WARRANTY; w

In [5]:
%load_ext autoreload
%autoreload 2
import pickle
from itertools import combinations

import numpy as np
import pandas as pd
import plotly.express as px
from pathlib import Path
from sklearn.model_selection import train_test_split

from quartz.data import MoleculeData, Grid, pki_grid_scale, scale_grids
from quartz.dataprocessing import (
    kmeans_clustering,
    dice_coefficient,
    calculate_pki_avg,
    spatial_agglomerative_clustering,
)
from quartz.grids import to_dxgrid


frontier_orbs_path = (
    "/scistor/gt/vie876/proj_FXa_QM/mwfn_frontier/results/frontier_grids.hdf5"
)
vdw_pot_path = "/scistor/gt/vie876/proj_FXa_QM/mwfn_vdw/results/vdw_grids.hdf5"
excel_path = Path("~/proj_FXa_QM/full_database_ligands.xlsx").expanduser()

In [6]:
def plot_cluster_overlap(
    cluster_grid,
    ligand_grids,
    avg_grid,
    props,
    weight_power=1,
    clusters_plotted=None,
    title=None,
):
    if title is None:
        _title = "auto"
    for key in props:
        overlap_per_cluster: dict[str, list[float]] = {}
        labels = cluster_grid.npGrid[key]

        if clusters_plotted is None:
            clusters_plotted = range(np.min(labels), np.max(labels) + 1)

        for i in clusters_plotted:
            indices = labels == i

            masked_grids = [
                g.npGrid[key][indices] * (g.weight**weight_power) for g in ligand_grids
            ]
            if not masked_grids:
                continue

            masked_grids_obj = [
                Grid(npGrid={key: m}, grid_props=g.grid_props)
                for g, m in zip(ligand_grids, masked_grids)
            ]

            masked_avg = avg_grid.npGrid[key][indices]
            masked_avg_obj = Grid(
                npGrid={key: masked_avg}, grid_props=ligand_grids[0].grid_props
            )

            pki_dices = dice_coefficient(masked_avg_obj, masked_grids_obj)
            overlap_per_cluster[f"pki_dice_{i}"] = list(pki_dices[key].values())

        df = pd.DataFrame(overlap_per_cluster)
        df["weight"] = [g.weight for g in ligand_grids]
        df["code"] = [g.mol_data.pdb_code for g in ligand_grids]
        df["doi"] = [g.mol_data.extra["doi"] for g in ligand_grids]
        df.dropna(inplace=True, axis=1)

        pki_cols = [col for col in df.columns if col.startswith("pki_dice_")]

        for cluster_col in pki_cols:
            corr = np.round(df[cluster_col].corr(df["weight"]), 3)
            cluster_id = cluster_col.split("_")[-1]

            if _title and _title == "auto":
                title = f"Dice with averaged pKi grids ({key}) | Cluster {cluster_id} | Corr: {corr}"

            fig = px.scatter(
                df,
                x=cluster_col,
                y="weight",
                hover_data=["code"],
                title=title,
                labels={
                    cluster_col: f"Dice with cluster {cluster_id}",
                    "weight": "weight",
                },
                color="doi",
                width=1500,
                height=900,
            )
            fig.show()


def export_cluster_masks(
    cluster_grid: Grid, prop: str, prefix: str, background_label: int = -1
) -> None:
    labels = cluster_grid.npGrid[prop]
    for label_id in sorted(np.unique(labels)):
        if label_id == background_label:
            continue
        mask = np.zeros_like(labels, dtype=np.float32)
        mask[labels == label_id] = 1.0
        to_dxgrid(
            mask,
            origin=cluster_grid.grid_props["origin"],
            spacing=cluster_grid.grid_props["size"],
            filename=f"{prefix}_{prop}_label{label_id}.dx",
        )


def combs_overlap(
    n_clusters: int,
    comb_len: int,
    props: list[str],
    cluster_grid: Grid,
    avg_grid: Grid,
    mol_grids: list[Grid],
):
    combs_overlap = {}
    combs = combinations(range(n_clusters), comb_len)

    for comb in combs:
        for key in props:
            labels = cluster_grid.npGrid[key]
            indices = np.isin(labels, comb)
            masked_mol_grids = [
                Grid(npGrid={key: g.npGrid[key][indices]}, grid_props=g.grid_props)
                for g in mol_grids
            ]
            masked_ref_grid = Grid(
                npGrid={key: avg_grid.npGrid[key][indices]},
                grid_props=avg_grid.grid_props,
            )
            dices = dice_coefficient(masked_ref_grid, masked_mol_grids)
            combs_overlap[str(comb)] = list(dices[key].values())

    overlap_df = pd.DataFrame(combs_overlap)
    overlap_df["pKi"] = [g.pki for g in mol_grids]

    corr_rows = []
    for comb_col in overlap_df.columns:
        if comb_col == "pKi":
            continue

        corr_val = overlap_df[comb_col].corr(overlap_df["pKi"])

        comb_ids = [x.strip() for x in comb_col.strip("()").split(",") if x.strip()]
        comb_label = "+".join(str(int(x)) for x in comb_ids)

        corr_rows.append(
            {
                "cluster_combination": comb_label,
                "corr_with_pKi": np.round(corr_val, 3),
            }
        )

    combs_overlap = (
        pd.DataFrame(corr_rows)
        .sort_values("corr_with_pKi", ascending=False)
        .reset_index(drop=True)
    )

    return combs_overlap


def plot_voxel_clusters(
    cluster_grid: Grid,
    prop: str,
    max_points: int = 50_000,
    background_label: int = -1,
):
    coords = np.argwhere(np.ones_like(cluster_grid.npGrid[prop], dtype=bool))
    label_values = cluster_grid.npGrid[prop][tuple(coords.T)]

    if len(coords) > max_points:
        sampled = np.random.default_rng(42).choice(
            len(coords), size=max_points, replace=False
        )
        coords = coords[sampled]
        label_values = label_values[sampled]

    fig = px.scatter_3d(
        x=coords[:, 0],
        y=coords[:, 1],
        z=coords[:, 2],
        color=label_values.astype(str),
        title=f"Clustering for {prop}",
        labels={"x": "X", "y": "Y", "z": "Z", "color": "Cluster"},
        opacity=0.55,
        width=1500,
        height=900,
    )
    fig.update_traces(marker={"size": 2})
    fig.show()

def comb_clusters_overlap(
    cluster_grid,
    mol_grids,
    avg_grid,
    props,
    clusters_to_combine: list[list[int]]
):
    """
    Calculates Dice coefficients for multiple arbitrary sets of combined clusters 
    and the corresponding non-scaled grids. 
    """
    dfs = {}
    
    for key in props:
        df = pd.DataFrame()
        df["pKi"] = [g.pki for g in mol_grids]
        df["code"] = [g.mol_data.pdb_code for g in mol_grids]
        df["doi"] = [g.mol_data.extra.get("doi", "missing") for g in mol_grids]
        
        for cluster_combo in clusters_to_combine:
            combined_label = "+".join(str(c) for c in cluster_combo)
            
            # mask the specified clusters
            labels = cluster_grid.npGrid[key]
            indices = np.isin(labels, cluster_combo)

            # mask the grids
            masked_grids = [g.npGrid[key][indices] for g in mol_grids]
            masekd_grids_obj = [
                Grid(npGrid={key: m}, grid_props=g.grid_props)
                for g, m in zip(mol_grids, masked_grids)
            ]

            # compare against weighted average grid
            pki_prop = avg_grid.npGrid[key]
            masked_pki = pki_prop[indices]
            masked_pki_obj = Grid(
                npGrid={key: masked_pki}, grid_props=mol_grids[0].grid_props
            )
            
            pki_dices = dice_coefficient(masked_pki_obj, masekd_grids_obj)
            
            df[f"pki_dice_{combined_label}"] = list(pki_dices[key].values())
            
        df.dropna(inplace=True)
        dfs[key] = df
        
    return dfs


def plot_comb_clusters_overlap(dfs, clusters_to_combine):
    """
    Plots the overlaps calculated by comb_clusters_overlap and returns
    a dataframe of the per-DOI correlations.
    """
    corr_per_doi_rows = []
    
    for prop, df in dfs.items():
        pki_cols = [col for col in df.columns if col.startswith("pki_dice_")]

        for cluster_col in pki_cols:
            combined_label = cluster_col.replace("pki_dice_", "")
            corr = df[cluster_col].corr(df["pKi"]).round(2)
            fig = px.scatter(
                df,
                x=cluster_col,
                y="pKi",
                hover_data=["code"],
                title=f"Dice with averaged pKi grids ({prop}) | Corr: {corr}",
                labels={
                    cluster_col: f"Dice with combined clusters [{combined_label}]",
                    "pKi": "pKi",
                },
                color="doi",
                width=1500,
                height=900,
            )
            fig.show()

            doi_counts = df.groupby("doi").size()
            valid_dois = doi_counts[doi_counts > 5].index

            for doi in valid_dois:
                doi_df = df[df["doi"] == doi]
                if doi_df[cluster_col].nunique() < 2 or doi_df["pKi"].nunique() < 2:
                    doi_corr = np.nan
                else:
                    doi_corr = doi_df[cluster_col].corr(doi_df["pKi"])

                corr_per_doi_rows.append(
                    {
                        "property": prop,
                        "cluster_col": cluster_col,
                        "doi": doi,
                        "n_compounds": int(len(doi_df)),
                        "corr": doi_corr,
                    }
                )

    corr_per_doi = pd.DataFrame(corr_per_doi_rows)
    if not corr_per_doi.empty:
        corr_per_doi = corr_per_doi.sort_values(
            by=["property", "cluster_col", "n_compounds"], ascending=[True, True, False]
        ).reset_index(drop=True)

    return corr_per_doi

In [7]:
nclusters = 8
props = ["polarizability"]

In [8]:
fx_data = pd.read_excel(excel_path).fillna("missing")
doi_dict = {
    row["PDB ID"]: row["doi / pdf link"]
    for _, row in fx_data.iterrows()
    if row["doi / pdf link"]
}

grids = pickle.load(open("grids_12.pkl", "rb"))


grids_with_pki: list[Grid] = [grid for grid in grids if not np.isnan(grid.pki)]
grids_without_pki: list[Grid] = [grid for grid in grids if np.isnan(grid.pki)]
# single_weighted = []
# double_weighted = scale_grids(single_weighted)

# grids_with_pki = []
single_weighted_abs = []
double_weighted_abs = []
single_weighted_rel = scale_grids(grids_with_pki)
double_weighted_rel = scale_grids(single_weighted_rel)
for grid in grids_with_pki:
    single_abs = pki_grid_scale(grid)
    single_weighted_abs.append(single_abs)
    double_weighted_abs.append(pki_grid_scale(single_abs))


pkis = [g.pki for g in grids_with_pki]

for grid in grids:
    grid.mol_data.extra["doi"] = doi_dict[grid.id_code]

pki_scaled_grids = [pki_grid_scale(grid) for grid in grids_with_pki]
to_cluster = [x for x in pki_scaled_grids if x is not None]

- Get compounds that have pKi set
- Set their weight to be pKi (does nothing I think as `pki_grid_scale()` uses `grid.pki`)
- Scale the grids according to their pKi
- Filter NaNs
- Cluster with KMeans

In [6]:
km_clusters = kmeans_clustering(to_cluster, props=props, n_clusters=nclusters)
agglomerative_clusters = spatial_agglomerative_clustering(
    to_cluster, props, n_clusters=nclusters
)

Here calculate the average grid on points with more than 50 contributions across all grids.
With `pre_scaled=False` grids are scaled by the pKi again resulting in $pKi^2$ scaling.

In [9]:
print("Calculating pKi average")
pki_avg = calculate_pki_avg(grids_with_pki, props=props, n_contribs=50, pre_scaled=False)

Calculating pKi average


In [10]:
aggl_clusters_pki = spatial_agglomerative_clustering([pki_avg], props, nclusters)
export_cluster_masks(aggl_clusters_pki, "polarizability", "pki_cluster")

Writing grids/data_pki_cluster_polarizability_label0.dx...
Writing grids/data_pki_cluster_polarizability_label1.dx...
Writing grids/data_pki_cluster_polarizability_label2.dx...
Writing grids/data_pki_cluster_polarizability_label3.dx...
Writing grids/data_pki_cluster_polarizability_label4.dx...
Writing grids/data_pki_cluster_polarizability_label5.dx...
Writing grids/data_pki_cluster_polarizability_label6.dx...
Writing grids/data_pki_cluster_polarizability_label7.dx...


In [19]:
grids[0].id_code

'0001'

In [20]:
for prop in props:
    to_dxgrid(
        pki_avg.npGrid[prop],
        origin=pki_avg.grid_props["origin"],
        filename=f"pki_avg_{prop}.dx",
    )

Writing grids/data_pki_avg_polarizability.dx...


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pca_result = pca.fit_transform(
    [grid.npGrid["polarizability"].flatten() for grid in pki_scaled_grids]
)
pca_df = pd.DataFrame(data=pca_result, columns=["PC1", "PC2"])
pca_df["pki"] = pkis
pca_df["code"] = [g.mol_data.pdb_code for g in actives]
pca_df["doi"] = [g.mol_data.extra["doi"] for g in actives]
fig = px.scatter(
    pca_df,
    x="PC1",
    y="PC2",
    color="doi",
    title="PCA of pKi Scaled Polarizability Grids",
    hover_data=["code", "doi"],
)
fig.show()

In [ ]:
from sklearn.cluster import KMeans

n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
pca_df["cluster"] = kmeans.fit_predict(pca_df[["PC1", "PC2"]])
fig = px.scatter(
    pca_df,
    x="PC1",
    y="PC2",
    color="cluster",
    title="PCA of pKi Scaled Polarizability Grids with KMeans Clusters",
    hover_data=["code", "pki"],
)
fig.show()

In [ ]:
clusters_plot = km_clusters
plot_props = ["polarizability"]  # , "RESP", "vdWpot"]

for key in plot_props:
    overlap_per_cluster: dict[str, list[float]] = {}
    labels_flat = clusters_plot.npGrid[key]

    for i in range(1, np.max(labels_flat) + 1):
        # print(f"Processing cluster {i}")
        indices = labels_flat == i
        masked_grids = [g.npGrid[key][indices] for g in pki_scaled_grids]
        masekd_grids_obj = [
            Grid(npGrid={key: m}, grid_props=g.grid_props)
            for g, m in zip(pki_scaled_grids, masked_grids)
        ]

        pki_prop = pki_avg.npGrid[key]
        masked_pki = pki_prop[indices]
        masked_pki_obj = Grid(
            npGrid={key: masked_pki}, grid_props=pki_scaled_grids[0].grid_props
        )

        masked_pki_grid = np.zeros_like(pki_prop)
        masked_pki_grid[indices] = masked_pki

        # Save masked pki to dx
        to_dxgrid(
            masked_pki_grid,
            filename="pki_cluster_" + str(i) + ".dx",
            origin=grids[0].grid_props["origin"],
            spacing=grids[0].grid_props["size"],
        )

        if not masked_grids:
            continue

        pki_dices = dice_coefficient(masked_pki_obj, masekd_grids_obj)

        overlap_per_cluster[f"pki_dice_{i}"] = list(pki_dices[key].values())

    df = pd.DataFrame(overlap_per_cluster)
    df["pKi"] = [g.pki for g in pki_scaled_grids]
    df["code"] = [g.mol_data.pdb_code for g in pki_scaled_grids]
    df.dropna(inplace=True)

    pki_cols = [col for col in df.columns if col.startswith("pki_dice_")]

    for cluster_col in pki_cols:
        corr = np.round(df[cluster_col].corr(df["pKi"]), 3)
        cluster_id = cluster_col.split("_")[-1]
        fig = px.scatter(
            df,
            x=cluster_col,
            y="pKi",
            hover_data=["code"],
            title=f"Dice with averaged pKi grids ({key}) | Cluster {cluster_id} | Corr: {corr}",
            labels={
                cluster_col: f"Dice with cluster {cluster_id}",
                "pKi": "pKi",
            },
        )
        fig.show()

In [ ]:
# Control which clusters to plot (exclude cluster 0 by default)
clusters_to_plot = [
    1,
    2,
    3,
    4,
    5,
]  # Modify this list to control which clusters are displayed

# Get cluster labels (3D grid shape)
labels = clusters_plot.npGrid["polarizability"]

# Generate grid coordinates based on the shape of the cluster data
shape = labels.shape
x_coords, y_coords, z_coords = np.meshgrid(
    np.arange(shape[0]), np.arange(shape[1]), np.arange(shape[2]), indexing="ij"
)

# Flatten all arrays for plotting
x_flat = x_coords.flatten()
y_flat = y_coords.flatten()
z_flat = z_coords.flatten()
labels_flat = labels.flatten()

# Filter to only include selected clusters
mask = np.isin(labels_flat, clusters_to_plot)
x_filtered = x_flat[mask]
y_filtered = y_flat[mask]
z_filtered = z_flat[mask]
labels_filtered = labels_flat[mask]

# Create a plotly 3D scatter plot with discrete colors for clusters
fig = px.scatter_3d(
    x=x_filtered,
    y=y_filtered,
    z=z_filtered,
    color=labels_filtered.astype(str),  # Convert to string for discrete colors
    title="K-Means Clustering of Grid Points",
    labels={"x": "X", "y": "Y", "z": "Z", "color": "Cluster"},
    opacity=0.6,
)
fig.update_traces(marker=dict(size=2))
fig.update_layout(legend_title_text="Cluster")
fig.show()


def plot_voxel_clusters(labels, title, top_cluster_ids):
    shape = labels.shape
    x_coords, y_coords, z_coords = np.meshgrid(
        np.arange(shape[0]), np.arange(shape[1]), np.arange(shape[2]), indexing="ij"
    )

    labels_flat = labels.flatten()
    mask = np.isin(labels_flat, np.array(top_cluster_ids, dtype=np.int32))

    fig = px.scatter_3d(
        x=x_coords.flatten()[mask],
        y=y_coords.flatten()[mask],
        z=z_coords.flatten()[mask],
        color=labels_flat[mask].astype(str),
        title=title,
        labels={"x": "X", "y": "Y", "z": "Z", "color": "Cluster"},
        opacity=0.6,
    )
    fig.update_traces(marker=dict(size=2))
    fig.update_layout(legend_title_text="Cluster")
    return fig

In [24]:
plot_cluster_overlap(agglomerative_clusters, grids_with_pki, pki_avg, props, 0)

In [10]:
combo_overlaps_2 = combs_overlap(
    8, 2, ["polarizability"], agglomerative_clusters, pki_avg, grids_with_pki
)
combo_overlaps_3 = combs_overlap(
    8, 3, ["polarizability"], agglomerative_clusters, pki_avg, grids_with_pki
)

In [11]:
combo_overlaps_2.head(4)

,cluster_combination,corr_with_pKi
0,1+4,0.916
1,1+2,0.893
2,1+7,0.891
3,0+1,0.890


In [12]:
combo_overlaps_3.head(4)

,cluster_combination,corr_with_pKi
0,1+3+4,0.936
1,1+2+4,0.926
2,1+4+7,0.917
3,1+4+5,0.911


In [17]:
pairs = [[1, 4], [1, 2], [0, 1], [1, 3, 4], [1, 2, 4], [1, 4, 5]]
dfs = comb_clusters_overlap(
    cluster_grid=agglomerative_clusters, 
    mol_grids=grids_with_pki, 
    avg_grid=pki_avg, 
    props=["polarizability"], 
    clusters_to_combine=pairs
)
corr_df = plot_comb_clusters_overlap(dfs, pairs)
corr_df


,property,cluster_col,doi,n_compounds,corr
0,polarizability,pki_dice_0+1,https://doi.org/10.1021/jm0111346,120,0.866271
1,polarizability,pki_dice_0+1,10.1021/jm049187l,105,0.941899
2,polarizability,pki_dice_0+1,https://doi.org/10.1016/j.bmcl.2007.06.029,67,0.851454
3,polarizability,pki_dice_0+1,10.1021/jm070125f,56,0.910935
4,polarizability,pki_dice_0+1,https://doi.org/10.1016/j.bmcl.2010.06.126,54,0.542246
...,...,...,...,...,...
175,polarizability,pki_dice_1+4+5,10.1515/BC.2002.130,13,0.953505
176,polarizability,pki_dice_1+4+5,https://doi.org/10.1016/j.bmcl.2010.01.011,12,0.759878
177,polarizability,pki_dice_1+4+5,10.1002/anie.201107091,9,0.816408
178,polarizability,pki_dice_1+4+5,https://doi.org/10.1016/j.bmcl.2006.08.092,8,0.967284


In [13]:
export_cluster_masks(agglomerative_clusters, "polarizability", "aggl")

Writing grids/data_aggl_polarizability_label0.dx...
Writing grids/data_aggl_polarizability_label1.dx...
Writing grids/data_aggl_polarizability_label2.dx...
Writing grids/data_aggl_polarizability_label3.dx...
Writing grids/data_aggl_polarizability_label4.dx...
Writing grids/data_aggl_polarizability_label5.dx...
Writing grids/data_aggl_polarizability_label6.dx...
Writing grids/data_aggl_polarizability_label7.dx...


In [14]:
plot_voxel_clusters(agglomerative_clusters, "polarizability", "agglomerative")

TypeError: '>' not supported between instances of 'int' and 'str'

# Linear regression

In [ ]:
from sklearn import linear_model
from sklearn import gaussian_process
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.svm import SVR

from sklearn.preprocessing import PowerTransformer


X = df["pki_dice_1"].values.reshape(-1, 1)

y = df["pKi"].values.reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2137
)
# reg = linear_model.LinearRegression().fit(X_train, y_train)
reg = linear_model.LinearRegression().fit(X_train, y_train)
y_pred = reg.predict(X_test)
diffs = y_pred - y_test
linreg_df = pd.DataFrame(
    {
        "Actual": y_test.flatten(),
        "Predicted": y_pred.flatten(),
        "Differences": diffs.flatten(),
    }
)
print(f"Mean squared error: {mean_squared_error(y_test, y_pred):.2f}")
print(f"Coefficient of determination: {r2_score(y_test, y_pred):.2f}")
fig = px.scatter(
    linreg_df,
    color="Differences",
    x="Predicted",
    y="Actual",
    trendline="ols",
    title="Linear Regression: Predicted vs Actual",
)
fig.show()


In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Binarize pKi targets for ROC evaluation (active vs inactive)
pki_threshold = 6.0
y_test_bin = (y_test.flatten() >= pki_threshold).astype(int)

fpr, tpr, _ = roc_curve(y_test_bin, y_pred.flatten())
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"Linear Regression ROC (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random baseline")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve (pKi ≥ {pki_threshold})")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

# Export data for Dash visualization (run on cluster)

In [ ]:
"""
DATA PREPARATION FOR DASH VISUALIZATION
========================================
Run this cell on the cluster to produce dashboard/dash_data.pkl (~15-25 MB).
Copy that file to your local machine, place it next to app.py, then run:
    python quartz/tests/dashboard/app.py
"""
import base64
import io
import pickle
from itertools import combinations
from pathlib import Path
from sqlite3 import SQLITE_DBCONFIG_ENABLE_FKEY

import h5py
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import (
    AllChem,
    Draw,
    rdFMCS,
    rdFingerprintGenerator,
    rdRGroupDecomposition,
)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from quartz.data import Grid, pki_grid_scale, scale_grids, set_weights
from quartz.dataprocessing import (
    calculate_pki_avg,
    calculate_weighted_avg,
    dice_coefficient,
    kmeans_clustering,
    spatial_agglomerative_clustering,
)

# -- Paths -------------------------------------------------------------------
_FRONTIER_PATH = Path(
    "/scistor/gt/vie876/proj_FXa_QM/mwfn_frontier/results/frontier_grids.hdf5"
)
_VDW_PATH = Path("/scistor/gt/vie876/proj_FXa_QM/mwfn_vdw/results/vdw_grids.hdf5")
_EXCEL_PATH = Path("~/proj_FXa_QM/full_database_ligands.xlsx").expanduser()
_OUTPUT_DIR = Path("dashboard")
_OUTPUT_DIR.mkdir(exist_ok=True)

# -- Config ------------------------------------------------------------------
_NCLUSTERS = 5
_PROPS = ["polarizability"]
# Clusters 1..n_clusters-1 are "active" regions; cluster 0 is background
_ACTIVE_CLUSTER_IDS = list(range(0, _NCLUSTERS))  # [1, 2, 3, 4]

# Morgan + KMeans + MCS config
_FP_RADIUS = 2
_FP_BITS = 2048
_FP_K_MAX = 20
_FP_RANDOM_STATE = 42
_MCS_THRESHOLD = 0.7

# -- Load raw data ------------------------------------------------------------
print("Loading grids.pkl ...")
_grids_dash: list[Grid] = pickle.load(open("grids.pkl", "rb"))

# print("Loading frontier orbitals ...")
# with h5py.File(_FRONTIER_PATH, "r") as _fh:
#     for _g in _grids_dash:
#         _homo = _fh[_g.id_code]["HOMO"][()]
#         _homo[_homo < 1e-6] = 0.0
#         _g.npGrid["HOMO_dens"] = _homo
#         _lumo = _fh[_g.id_code]["LUMO"][()]
#         _lumo[_lumo < 1e-6] = 0.0
#         _g.npGrid["LUMO_dens"] = _lumo

# print("Loading vdW potential ...")
# with h5py.File(_VDW_PATH, "r") as _fh:
#     for _g in _grids_dash:
#         _vdw = _fh[_g.id_code]["vdWpot"][()]
#         _vdw[np.abs(_vdw) < 1e-5] = 0.0
#         _g.npGrid["vdWpot"] = _vdw

print("Loading DOI map ...")
_fx_db = pd.read_excel(_EXCEL_PATH).fillna("missing")
_doi_map = {row["PDB ID"]: row["doi / pdf link"] for _, row in _fx_db.iterrows()}
for _g in _grids_dash:
    _g.mol_data.extra["doi"] = _doi_map.get(_g.id_code, "missing")

# -- Actives + pKi-scaled grids ----------------------------------------------
print("Preparing actives ...")
_actives_dash = [g for g in _grids_dash if not np.isnan(g.pki)]
_pki_scaled = scale_grids(_actives_dash)
# _pki_scaled = [pki_grid_scale(g) for g in _actives_dash]
# _pki_scaled = [g for g in _pki_scaled if g is not None]
print(f"  {len(_pki_scaled)} active compounds")

# -- KMeans clustering --------------------------------------------------------
print("Running KMeans clustering (n_clusters=5) ...")
_km = spatial_agglomerative_clustering(_pki_scaled, _PROPS, _NCLUSTERS, "average")

# -- pKi-weighted average grid ------------------------------------------------
print("Computing pKi-weighted average grids ...")
_pki_avg_dash = calculate_weighted_avg(
    _pki_scaled, props=_PROPS, n_contribs=50, pre_weighed=False
)


# -- Dice for all non-empty subsets of active clusters -----------------------
def _subset_key(subset):
    return "+".join(str(c) for c in sorted(subset))


_all_subsets = [
    combo
    for r in range(1, len(_ACTIVE_CLUSTER_IDS) + 1)
    for combo in combinations(_ACTIVE_CLUSTER_IDS, r)
]

print(
    f"Computing Dice for {len(_PROPS)} props x {len(_all_subsets)} cluster subsets ..."
)
_dice_cols: dict[str, list[float]] = {}

for _prop in _PROPS:
    print(f"  {_prop} ...")
    _lbls = _km.npGrid[_prop]
    for _subset in _all_subsets:
        _col = f"dice_{_prop}_{_subset_key(_subset)}"
        _idx = np.isin(_lbls, _subset)
        _mg_obj = [
            Grid(npGrid={_prop: g.npGrid[_prop][_idx]}, grid_props=g.grid_props)
            for g in _pki_scaled
        ]
        _ref_obj = Grid(
            npGrid={_prop: _pki_avg_dash.npGrid[_prop][_idx]},
            grid_props=_pki_scaled[0].grid_props,
        )
        _dices = dice_coefficient(_ref_obj, _mg_obj)
        _dice_cols[_col] = list(_dices[_prop].values())

# -- Cluster-to-cluster Dice correlation matrix ------------------------------
print("Computing per-property cluster correlation matrices ...")
_cluster_corr: dict[str, np.ndarray] = {}
_n_active = len(_ACTIVE_CLUSTER_IDS)

for _prop in _PROPS:
    _single_cols = [f"dice_{_prop}_{c}" for c in _ACTIVE_CLUSTER_IDS]
    _tmp = pd.DataFrame({c: _dice_cols[c] for c in _single_cols}).dropna()
    if len(_tmp) > 1:
        _cluster_corr[_prop] = _tmp.corr().values
    else:
        _cluster_corr[_prop] = np.full((_n_active, _n_active), np.nan)

# -- Build compounds DataFrame ------------------------------------------------
print("Generating 2D RDKit images ...")


def _mol_to_b64(smiles: str) -> str:
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return ""
        AllChem.Compute2DCoords(mol)
        img = Draw.MolToImage(mol, size=(300, 200))
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        return base64.b64encode(buf.getvalue()).decode()
    except Exception:
        return ""


def _smiles_to_mol(smiles: str):
    if not isinstance(smiles, str) or smiles.strip() == "":
        return None
    try:
        return Chem.MolFromSmiles(smiles)
    except Exception:
        return None


def _pick_elbow_k(ks: list[int], inertias: list[float]) -> int:
    if len(ks) <= 2:
        return ks[0]
    _x = np.asarray(ks, dtype=float)
    _y = np.asarray(inertias, dtype=float)
    _x0, _y0 = _x[0], _y[0]
    _x1, _y1 = _x[-1], _y[-1]
    _den = np.hypot(_y1 - _y0, _x1 - _x0)
    if _den == 0:
        return ks[0]
    _dist = np.abs((_y1 - _y0) * _x - (_x1 - _x0) * _y + _x1 * _y0 - _y1 * _x0) / _den
    return int(ks[int(np.argmax(_dist))])


def _rgroup_col_sort(_name: str) -> tuple[int, str]:
    if _name.startswith("R") and _name[1:].isdigit():
        return (int(_name[1:]), _name)
    return (10**6, _name)


_compounds = pd.DataFrame(
    {
        "id_code": [g.mol_data.pdb_code for g in _pki_scaled],
        "pki": [g.pki for g in _pki_scaled],
        "smiles": [str(g.mol_data.smiles) for g in _pki_scaled],
        "doi": [g.mol_data.extra.get("doi", "missing") for g in _pki_scaled],
    }
)
_compounds["mol_img_b64"] = _compounds["smiles"].apply(_mol_to_b64)
for _col, _vals in _dice_cols.items():
    _compounds[_col] = _vals

# -- Choose overall overlap metric + simple OLS baseline ---------------------
_compounds["overlap_metric"] = np.nan
_compounds["ols_pred_overall"] = np.nan
_compounds["ols_error_overall"] = np.nan
_overlap_metric_name = ""

_dice_metric_cols = [c for c in _compounds.columns if c.startswith("dice_")]
if _dice_metric_cols:
    _corr = _compounds[_dice_metric_cols].corrwith(_compounds["pki"]).dropna()
    if not _corr.empty:
        _overlap_metric_name = str(_corr.abs().idxmax())
        _compounds["overlap_metric"] = pd.to_numeric(
            _compounds[_overlap_metric_name], errors="coerce"
        )
        _fit_df = _compounds[["overlap_metric", "pki"]].dropna()
        if len(_fit_df) >= 2 and _fit_df["overlap_metric"].nunique() > 1:
            _slope, _intercept = np.polyfit(
                _fit_df["overlap_metric"].to_numpy(dtype=float),
                _fit_df["pki"].to_numpy(dtype=float),
                1,
            )
            _pred = _compounds["overlap_metric"] * _slope + _intercept
            _compounds["ols_pred_overall"] = _pred
            _compounds["ols_error_overall"] = _compounds["pki"] - _pred

# -- Morgan fingerprints + KMeans clustering ---------------------------------
print("Computing Morgan fingerprints and KMeans clusters (elbow + silhouette) ...")
_morgan_gen = rdFingerprintGenerator.GetMorganGenerator(
    radius=_FP_RADIUS, fpSize=_FP_BITS
)
_mols = [_smiles_to_mol(_s) for _s in _compounds["smiles"]]
_fps = [(_morgan_gen.GetFingerprint(_m) if _m is not None else None) for _m in _mols]

_compounds["fp_valid"] = [fp is not None for fp in _fps]
_compounds["morgan_fp_bits"] = [
    fp.ToBitString() if fp is not None else "" for fp in _fps
]
_compounds["morgan_fp_n_onbits"] = [
    int(fp.GetNumOnBits()) if fp is not None else 0 for fp in _fps
]

_valid_global_idx = [i for i, fp in enumerate(_fps) if fp is not None]
_valid_fps = [_fps[i] for i in _valid_global_idx]
_fp_cluster_labels = np.full(len(_compounds), -1, dtype=int)
_k_eval_rows: list[dict[str, float]] = []
_selected_k = 1
_elbow_k = 1
_silhouette_k = 1

if len(_valid_fps) == 1:
    _fp_cluster_labels[_valid_global_idx[0]] = 0
elif len(_valid_fps) == 2:
    _fp_cluster_labels[_valid_global_idx] = 0
elif len(_valid_fps) > 2:
    _fp_matrix = np.array(
        [list(map(int, fp.ToBitString())) for fp in _valid_fps], dtype=np.uint8
    )
    _k_candidates = list(range(2, min(_FP_K_MAX, len(_valid_fps) - 1) + 1))

    _inertias = []
    _silhouettes = []
    for _k in _k_candidates:
        _km_fp = KMeans(n_clusters=_k, random_state=_FP_RANDOM_STATE, n_init=20)
        _labels_k = _km_fp.fit_predict(_fp_matrix)
        _inertias.append(float(_km_fp.inertia_))
        _silhouettes.append(float(silhouette_score(_fp_matrix, _labels_k)))

    _elbow_k = _pick_elbow_k(_k_candidates, _inertias)
    _silhouette_k = int(_k_candidates[int(np.argmax(_silhouettes))])
    _selected_k = _silhouette_k

    for _k, _inertia, _sil in zip(_k_candidates, _inertias, _silhouettes):
        _k_eval_rows.append(
            {
                "k": int(_k),
                "inertia": float(_inertia),
                "silhouette": float(_sil),
                "is_elbow": bool(_k == _elbow_k),
                "is_silhouette_best": bool(_k == _silhouette_k),
                "is_selected": bool(_k == _selected_k),
            }
        )

    _km_final = KMeans(n_clusters=_selected_k, random_state=_FP_RANDOM_STATE, n_init=20)
    _final_labels = _km_final.fit_predict(_fp_matrix)
    for _local_idx, _lbl in enumerate(_final_labels):
        _fp_cluster_labels[_valid_global_idx[_local_idx]] = int(_lbl)

_compounds["fp_cluster"] = _fp_cluster_labels
_fp_kmeans_eval = pd.DataFrame(_k_eval_rows)

# -- MCS per fingerprint cluster (support threshold = 0.7) ------------------
print("Computing per-cluster MCS (threshold=0.7) ...")
_mcs_rows = []
_cluster_sizes = (
    _compounds.loc[_compounds["fp_cluster"] >= 0, "fp_cluster"]
    .value_counts()
    .sort_index()
    .to_dict()
)

for _cid, _size in sorted(_cluster_sizes.items()):
    _member_idx = _compounds.index[_compounds["fp_cluster"] == _cid].tolist()
    _cluster_mols = [_mols[i] for i in _member_idx if _mols[i] is not None]

    if len(_cluster_mols) < 2:
        _mcs_rows.append(
            {
                "fp_cluster": int(_cid),
                "n_members": int(_size),
                "mcs_smarts": "",
                "mcs_num_atoms": 0,
                "mcs_num_bonds": 0,
                "mcs_match_count": 0,
                "mcs_status": "insufficient_molecules",
            }
        )
        continue

    try:
        _mcs_res = rdFMCS.FindMCS(
            _cluster_mols,
            threshold=_MCS_THRESHOLD,
            timeout=60,
            ringMatchesRingOnly=True,
            completeRingsOnly=True,
        )
        _smarts = _mcs_res.smartsString if _mcs_res.numAtoms > 0 else ""
        if _smarts:
            _qmol = Chem.MolFromSmarts(_smarts)
            _matches = (
                sum(int(_m.HasSubstructMatch(_qmol)) for _m in _cluster_mols)
                if _qmol is not None
                else 0
            )
            _status = "ok" if not _mcs_res.canceled else "timeout_partial"
        else:
            _matches = 0
            _status = "no_mcs"

        _mcs_rows.append(
            {
                "fp_cluster": int(_cid),
                "n_members": int(_size),
                "mcs_smarts": _smarts,
                "mcs_num_atoms": int(_mcs_res.numAtoms),
                "mcs_num_bonds": int(_mcs_res.numBonds),
                "mcs_match_count": int(_matches),
                "mcs_status": _status,
            }
        )
    except Exception as _exc:
        _mcs_rows.append(
            {
                "fp_cluster": int(_cid),
                "n_members": int(_size),
                "mcs_smarts": "",
                "mcs_num_atoms": 0,
                "mcs_num_bonds": 0,
                "mcs_match_count": 0,
                "mcs_status": f"error: {_exc}",
            }
        )

_fp_cluster_mcs = pd.DataFrame(_mcs_rows)
if _fp_cluster_mcs.empty:
    _fp_cluster_mcs = pd.DataFrame(
        columns=[
            "fp_cluster",
            "n_members",
            "mcs_smarts",
            "mcs_num_atoms",
            "mcs_num_bonds",
            "mcs_match_count",
            "mcs_status",
        ]
    )

_mcs_map = {
    int(row["fp_cluster"]): row["mcs_smarts"]
    for _, row in _fp_cluster_mcs.iterrows()
    if int(row["fp_cluster"]) >= 0
}
_compounds["fp_cluster_mcs_smarts"] = [
    _mcs_map.get(int(_cid), "") if int(_cid) >= 0 else ""
    for _cid in _compounds["fp_cluster"]
]
_compounds["fp_cluster_size"] = [
    int(_cluster_sizes.get(int(_cid), 0)) for _cid in _compounds["fp_cluster"]
]

_fp_cluster_summary = (
    _compounds.loc[_compounds["fp_cluster"] >= 0, ["fp_cluster", "id_code", "doi"]]
    .groupby("fp_cluster", as_index=False)
    .agg(
        n_members=("id_code", "size"),
        n_unique_doi=("doi", "nunique"),
    )
    .sort_values("fp_cluster")
    .reset_index(drop=True)
)

# -- R-group decomposition using per-cluster MCS -----------------------------
print("Running R-group decomposition per fingerprint cluster ...")
_rgroup_rows: list[dict[str, object]] = []

for _cid, _size in sorted(_cluster_sizes.items()):
    _member_idx = _compounds.index[_compounds["fp_cluster"] == _cid].tolist()
    _cluster_mols = [_mols[i] for i in _member_idx if _mols[i] is not None]
    _mcs_smarts = _mcs_map.get(int(_cid), "")

    if len(_cluster_mols) < 2 or not _mcs_smarts:
        for _gidx in _member_idx:
            _row = _compounds.loc[_gidx]
            _rgroup_rows.append(
                {
                    "id_code": _row["id_code"],
                    "fp_cluster": int(_row["fp_cluster"]),
                    "pki": float(_row["pki"]),
                    "overlap_metric": _row["overlap_metric"],
                    "overlap_metric_name": _overlap_metric_name,
                    "ols_pred_overall": _row["ols_pred_overall"],
                    "ols_error_overall": _row["ols_error_overall"],
                    "core": "",
                    "rgroup_status": "no_mcs",
                }
            )
        continue

    _core_mol = Chem.MolFromSmarts(_mcs_smarts)
    if _core_mol is None:
        for _gidx in _member_idx:
            _row = _compounds.loc[_gidx]
            _rgroup_rows.append(
                {
                    "id_code": _row["id_code"],
                    "fp_cluster": int(_row["fp_cluster"]),
                    "pki": float(_row["pki"]),
                    "overlap_metric": _row["overlap_metric"],
                    "overlap_metric_name": _overlap_metric_name,
                    "ols_pred_overall": _row["ols_pred_overall"],
                    "ols_error_overall": _row["ols_error_overall"],
                    "core": "",
                    "rgroup_status": "invalid_mcs_core",
                }
            )
        continue

    try:
        _rg_rows, _rg_unmatched = rdRGroupDecomposition.RGroupDecompose(
            [_core_mol],
            _cluster_mols,
            asRows=True,
            asSmiles=True,
        )
        _rg_unmatched = set(_rg_unmatched)
        _rg_iter = iter(_rg_rows)

        for _local_idx, _gidx in enumerate(_member_idx):
            _row = _compounds.loc[_gidx]
            _base = {
                "id_code": _row["id_code"],
                "fp_cluster": int(_row["fp_cluster"]),
                "pki": float(_row["pki"]),
                "overlap_metric": _row["overlap_metric"],
                "overlap_metric_name": _overlap_metric_name,
                "ols_pred_overall": _row["ols_pred_overall"],
                "ols_error_overall": _row["ols_error_overall"],
            }

            if _local_idx in _rg_unmatched:
                _base["core"] = _mcs_smarts
                _base["rgroup_status"] = "unmatched"
                _rgroup_rows.append(_base)
                continue

            _decomp = next(_rg_iter, {})
            _base["core"] = str(_decomp.get("Core", _mcs_smarts) or _mcs_smarts)
            for _k, _v in _decomp.items():
                if _k == "Core":
                    continue
                _base[_k] = "" if _v is None else str(_v)
            _base["rgroup_status"] = "ok"
            _rgroup_rows.append(_base)
    except Exception as _exc:
        for _gidx in _member_idx:
            _row = _compounds.loc[_gidx]
            _rgroup_rows.append(
                {
                    "id_code": _row["id_code"],
                    "fp_cluster": int(_row["fp_cluster"]),
                    "pki": float(_row["pki"]),
                    "overlap_metric": _row["overlap_metric"],
                    "overlap_metric_name": _overlap_metric_name,
                    "ols_pred_overall": _row["ols_pred_overall"],
                    "ols_error_overall": _row["ols_error_overall"],
                    "core": _mcs_smarts,
                    "rgroup_status": f"error: {_exc}",
                }
            )

for _gidx in _compounds.index[_compounds["fp_cluster"] < 0].tolist():
    _row = _compounds.loc[_gidx]
    _rgroup_rows.append(
        {
            "id_code": _row["id_code"],
            "fp_cluster": int(_row["fp_cluster"]),
            "pki": float(_row["pki"]),
            "overlap_metric": _row["overlap_metric"],
            "overlap_metric_name": _overlap_metric_name,
            "ols_pred_overall": _row["ols_pred_overall"],
            "ols_error_overall": _row["ols_error_overall"],
            "core": "",
            "rgroup_status": "no_fp_cluster",
        }
    )

_fp_cluster_rgroup = pd.DataFrame(_rgroup_rows)
if _fp_cluster_rgroup.empty:
    _fp_cluster_rgroup = pd.DataFrame(
        columns=[
            "id_code",
            "fp_cluster",
            "pki",
            "overlap_metric",
            "overlap_metric_name",
            "ols_pred_overall",
            "ols_error_overall",
            "core",
            "rgroup_status",
        ]
    )
else:
    _base_cols = [
        "id_code",
        "fp_cluster",
        "pki",
        "overlap_metric",
        "overlap_metric_name",
        "ols_pred_overall",
        "ols_error_overall",
        "core",
    ]
    _r_cols = sorted(
        [c for c in _fp_cluster_rgroup.columns if c.startswith("R")],
        key=_rgroup_col_sort,
    )
    _tail_cols = ["rgroup_status"]
    _other = [
        c
        for c in _fp_cluster_rgroup.columns
        if c not in _base_cols + _r_cols + _tail_cols
    ]
    _fp_cluster_rgroup = _fp_cluster_rgroup[_base_cols + _r_cols + _other + _tail_cols]
    _fp_cluster_rgroup = _fp_cluster_rgroup.sort_values(
        ["fp_cluster", "id_code"]
    ).reset_index(drop=True)

# -- Save bundle --------------------------------------------------------------
_bundle = {
    "compounds": _compounds,
    "cluster_labels": {p: _km.npGrid[p] for p in _PROPS},
    "cluster_corr_matrix": _cluster_corr,
    "props": _PROPS,
    "cluster_ids": _ACTIVE_CLUSTER_IDS,
    "n_clusters": _NCLUSTERS,
    "grid_shape": _km.npGrid[_PROPS[0]].shape,
    "subset_keys": [_subset_key(s) for s in _all_subsets],
    "fingerprint_settings": {
        "method": "morgan",
        "radius": _FP_RADIUS,
        "n_bits": _FP_BITS,
        "cluster_method": "kmeans_fingerprint_bits",
        "k_max": _FP_K_MAX,
        "k_random_state": _FP_RANDOM_STATE,
        "k_selected": int(_selected_k),
        "k_elbow": int(_elbow_k),
        "k_silhouette": int(_silhouette_k),
        "mcs_threshold": _MCS_THRESHOLD,
        "overlap_metric_name": _overlap_metric_name,
    },
    "fingerprint_kmeans_eval": _fp_kmeans_eval,
    "fingerprint_cluster_summary": _fp_cluster_summary,
    "fingerprint_cluster_mcs": _fp_cluster_mcs,
    "fingerprint_cluster_rgroup": _fp_cluster_rgroup,
}

_out = _OUTPUT_DIR / "dash_data.pkl"
with open(_out, "wb") as _fh:
    pickle.dump(_bundle, _fh)

_mb = _out.stat().st_size / 1e6
print(f"\nSaved -> {_out}  ({_mb:.1f} MB)")
print(f"Compounds: {_compounds.shape}")
print(f"Dice columns: {len(_dice_cols)}")
print(f"Overlap metric: {_overlap_metric_name if _overlap_metric_name else 'n/a'}")
print(f"Valid fingerprints: {int(_compounds['fp_valid'].sum())} / {len(_compounds)}")
print(
    f"Fingerprint clusters: {int(_compounds.loc[_compounds['fp_cluster'] >= 0, 'fp_cluster'].nunique())}"
)
print(
    f"KMeans selection (elbow/silhouette/final): {_elbow_k}/{_silhouette_k}/{_selected_k}"
)
print(f"Clusters with MCS rows: {len(_fp_cluster_mcs)}")
print(f"R-group rows: {len(_fp_cluster_rgroup)}")
print(
    f"Cluster label shapes: { {p: v.shape for p, v in _bundle['cluster_labels'].items()} }"
)
print("\nTo run the Dash app on a local machine:")
print("  1. scp this_server:/path/to/dashboard/dash_data.pkl ./quartz/tests/dashboard/")
print("  2. pip install dash dash-bootstrap-components plotly pandas numpy")
print("  3. python quartz/tests/dashboard/app.py")
